# PastTheEyeTest 2027 projections — validation & graphics

This notebook refits the projection model from your history files and produces every table and chart for the article. It imports the **same code the site's build uses** (`build_projections.py`), so the numbers here match the published 2027 projections.

**To run:** Runtime → Run all. If this copy has the two hidden "embedded" cells at the top, the 2015–2026 history and `article_tools.py` are already inside the notebook, so there's nothing to upload. This copy contains your private data, so don't share it publicly.

**To refresh the data later**, delete the two embedded cells and Run all. The first cell then asks you to upload:
1. a new `projection_bundle.tar.gz`, built on the droplet with:
   ```
   cd ~/ptet-proj && tar czf ~/projection_bundle.tar.gz --ignore-failed-read build_projections.py history/hitter_history.csv history/pitcher_history.csv history/birthdates.json history/iswing_history.csv private_out/baseball_data_2027P.json private_out/hitter_xrv_2027P.json
   ```
2. `article_tools.py`.

Everything lands in `figures/` (PNG, 200 dpi) and `tables/` (CSV); the last cell zips both for download. Runtime: about 5–10 minutes on the full 2015–2026 history.

The formulas behind every step are in `PROJECTION_FORMULAS.md`.

In [ ]:
# 1 · Unpack the data (asks for an upload only if nothing is embedded / present) + article_tools.py (skipped automatically outside Colab)
import glob, os, re, shutil, tarfile, zipfile
try:
    from google.colab import files
except ImportError:
    files = None
have = glob.glob("*.tar.gz") + glob.glob("projection_bundle*.zip")
uploaded = files.upload() if (files and not (have and os.path.exists("article_tools.py"))) else {}
for f in list(uploaded) + glob.glob("*.tar.gz") + glob.glob("*.zip"):
    clean = re.sub(r" \(\d+\)", "", f)          # Chromebook adds " (1)" to repeat downloads
    if clean != f and os.path.exists(f):
        shutil.move(f, clean); f = clean
    if f.endswith(".tar.gz"):
        tarfile.open(f).extractall(".")
    elif f.endswith(".zip") and f != "article_outputs.zip":
        zipfile.ZipFile(f).extractall(".")
for f in glob.glob("* (*).py"):
    shutil.move(f, re.sub(r" \(\d+\)", "", f))
for need in ["build_projections.py", "article_tools.py", "history/hitter_history.csv", "history/pitcher_history.csv",
             "history/birthdates.json", "history/iswing_history.csv", "private_out/baseball_data_2027P.json"]:
    print(("✓ " if os.path.exists(need) else "✗ missing ") + need)

In [ ]:
# 2 · Settings
HISTORY = "history"        # hitter_history.csv, pitcher_history.csv, birthdates.json, iswing_history.csv
PUBLIC  = "private_out"    # baseball_data_2027P.json / hitter_xrv_2027P.json (for the player range charts)
HITTER_LAYERS = os.path.exists(f"{HISTORY}/iswing_history.csv")   # re-run the iSwing+ layer check too

import sys; sys.path.insert(0, ".")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import importlib, article_tools as at
importlib.reload(at)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

## Part 1 · The data
One row per player-season. Hitters' sample sizes are PA, balls in play (BIP) and swings; pitchers' are batters faced (BF), innings and pitches. Each stat uses the first of its preferred sample sizes that is filled on at least 90% of its rows.

In [ ]:
H = at.fit_side("hitters", history=HISTORY, public=PUBLIC, layers=HITTER_LAYERS)
P = at.fit_side("pitchers", history=HISTORY, public=PUBLIC, layers=True)
display(at.table_data(H).merge(at.table_data(P), on="season", how="outer", suffixes=(" hitters", " pitchers")))

## Part 2 · The model: season weights, regression and aging
For every stat, the model learns from the history:

$$\hat{x}_{T} = \overline{x}_{T-1} + \frac{\sum_{j=1}^{J} w_j\, n_{T-j}\,\big(x_{T-j}-\overline{x}_{T-j}\big)}{\sum_{j=1}^{J} w_j\, n_{T-j} + k} + a_1(\text{age}-28) + a_2(\text{age}-28)^2$$

* $w_j$ — weight of the season $j$ years back ($w_1 = 1$; older seasons never count more than newer ones)
* $k$ — regression constant, in the stat's own sample units: after one season of $n$, his own record carries $n/(n+k)$ of the projection
* $a_1, a_2$ — the stat's aging curve, fitted at the same time as the weights so they don't absorb aging

In [ ]:
display(at.table_model(H))
display(at.table_model(P))

In [ ]:
at.plot_weights(H); at.plot_weights(P); plt.show()

**What a projection is made of.** For a full-time player with three straight full seasons ($n$ = the 90th percentile of last season's qualified samples), the share from each season is $w_j n/(\sum w n + k)$ and the league average gets $k/(\sum w n + k)$.

In [ ]:
at.plot_composition(H); at.plot_composition(P); plt.show()

**How fast each stat earns trust:** $n/(n+k)$ — the weight on one season of his own data as that season's sample grows. Stats that stabilise fast (bat speed, exit velocity) climb steeply; noisy ones (BB%, pitch-type xRV) stay low.

In [ ]:
at.plot_shrinkage(H, ["Avg Bat Speed", "90th % EV", "Whiff %", "K%", "Chase %", "xSLG", "Barrel %", "BB%", "xRV/600", "LA+SwtSpt%"])
at.plot_shrinkage(P, ["Avg FB Velo", "Whiff%", "K%", "Chase%", "BB%", "GB%", "xwOBA", "FIP", "Barrel%"])
plt.show()

**Does the fitted $k$ agree with how sticky each stat really is?** Regression to the mean keeps $n/(n+k)$ of a player's gap from average, so for the typical sample $n$ that share should track the year-to-year correlation $r$.

In [ ]:
for S in (H, P):
    display(at.table_stability(S)); at.plot_stability(S)
plt.show()

**Aging.** Each stat's curve $a_1(\text{age}-28) + a_2(\text{age}-28)^2$, shown in league standard deviations so stats are comparable (up = better). Pitcher ages are clamped to 22–37.

In [ ]:
at.plot_aging(H); at.plot_aging(P); plt.show()

## Part 3 · Validation: walk-forward backtest
Each of the last three seasons $T$ is projected using **only data from before $T$**, then scored against what actually happened. The error is sample-weighted:

$$\text{RMSE} = \sqrt{\frac{\sum_i n_i\,(\hat{x}_i - x_i)^2}{\sum_i n_i}}$$

Compared methods: league average; last season's value; Marcel's fixed 5/4/3/2/1 weights (with $k$ still fitted); the learned model. Rows are players with a last season, so all methods see the same players.

In [ ]:
display(at.table_backtest(H)); display(at.table_backtest(P))
at.plot_backtest(H); at.plot_backtest(P); plt.show()

In [ ]:
at.plot_backtest_scatter(H, ["xSLG", "xwOBACON", "K%", "BB%", "Chase %", "Avg Bat Speed", "Barrel %", "xRV/600"])
at.plot_backtest_scatter(P, ["K%", "BB%", "Whiff%", "xwOBA", "FIP", "Avg FB Velo", "GB%", "Barrel%"])
plt.show()

## Part 4 · Validation: model layers (iSwing+, Stuff+/Location+/Tunnel+/Pitch+)
A layer stacks projected model grades onto the base projection, $\hat{y} = c_0\,\hat{y}_\text{base} + \sum_m c_m \hat{s}_m$ (no intercept, sample-weighted least squares). It is used **only** if leave-one-season-out error beats the base model by at least 0.2%.

In [ ]:
display(at.table_layers(H)); display(at.table_layers(P))

## Part 5 · Validation: the lean correction
Regressing everyone to the playing-time-weighted league average pulls thin-history players toward what regulars do — and thin-history players are worse on average, so they were projected too high. The fix adds

$$\beta\,\frac{k}{H+k}, \qquad H = \sum_j w_j\, n_{T-j}$$

largest for the thinnest history and ≈0 for a long full-time career. $\beta$ is fit on the backtest misses and **kept only if** it lowers the walk-forward error in at least two-thirds of the tests.

In [ ]:
display(at.table_lean(H)); display(at.table_lean(P))

In [ ]:
# Hitters: the applied corrections with the biggest error cut, plus K% as one it didn't earn
for s in at.lean_examples(H) + ["K%"]:
    at.plot_lean(H, s)
# Pitchers: no stat may earn it; then this shows the largest β as a counter-example
for s in at.lean_examples(P):
    at.plot_lean(P, s)
plt.show()

## Part 6 · Validation: are the 2027 ranges honest?
The ranges come from the walk-forward misses themselves (on the logit scale for rates):

$$\sigma^2 = c_0 + \frac{a}{H+k} + \frac{b}{n} \qquad \text{(non-negative least squares)}$$

Each backtest season's misses are standardised, $z = \text{miss}/\sigma$, by a spread fit on the **other** seasons, and the band edges are the empirical percentiles of $z$ — no bell-curve assumption. The check below scores each season with bands fit only on the other seasons: a calibrated 90% band contains ≈90% of outcomes.

In [ ]:
display(at.table_coverage(H)); display(at.table_coverage(P))
at.plot_coverage(H); at.plot_coverage(P)
at.plot_coverage_heatmap(H); at.plot_coverage_heatmap(P); plt.show()

In [ ]:
for s in ["K%", "xSLG", "BB%"]:
    at.plot_spread(H, s); at.plot_miss_shape(H, s)
for s in ["K%", "xwOBA"]:
    at.plot_spread(P, s); at.plot_miss_shape(P, s)
plt.show()

## Part 7 · Player range charts
The Ranges tab's picture for any player, read from the published 2027 projection file: every stat on one axis of league standard deviations from last season's average (right = better), nested 90 / 75 / 50 / 25% ranges, the projection (dot) and last season (ring).

In [ ]:
proj = at.load_projection(PUBLIC)
at.plot_fan(proj, "Aaron Judge")
at.plot_fan(proj, "Aaron Judge", xrv=True)
at.plot_fan(proj, "Tarik Skubal", side="pitchers")
plt.show()

## Part 8 · (Optional) Full build log
Reruns the real builds into a scratch folder and prints their tables: weights, $k$, backtests, layer choices, lean fixes and calibration per stat. It doesn't touch your `private_out`.

In [ ]:
RUN_FULL_BUILD = False
if RUN_FULL_BUILD:
    import subprocess
    run = subprocess.run([sys.executable, "build_projections.py", "--public", "rebuild_check", "--history", HISTORY],
                         capture_output=True, text=True)
    print(run.stdout, run.stderr[-3000:])

## Download everything

In [ ]:
with zipfile.ZipFile("article_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d in ("figures", "tables"):
        for f in sorted(glob.glob(f"{d}/*")):
            z.write(f)
print(f"{len(glob.glob('figures/*'))} figures, {len(glob.glob('tables/*'))} tables → article_outputs.zip")
if files:
    files.download("article_outputs.zip")